# Set up an HMS-HMS grid with a POSYDON rerun

<div class='alert alert-info'>

In this how-to you set up a grid of binaries with two hydrogen-rich stars
(HMS-HMS) using the physics of the POSYDON v2.0 release, *as changed by
one of its reruns*. You end with a run directory that is ready to submit
to a SLURM cluster, and a record of exactly which files it was built
from.

</div>

The POSYDON v2.0 grids were not all run with the same inlists: after the
first runs, several *reruns* changed the physics for some of the grid
(for example, to remove the age limit or to change the LBV winds). Each
rerun is a named folder in POSYDON-MESA-INLISTS that holds only what it
changes. To set up a grid with a rerun's physics you name the release and
the rerun in your `ini` file; `posydon-grid` stacks the rerun on top of
the release.

After getting MESA and the POSYDON inlists ready, the steps are:

1. Choose a rerun.
2. Write the grid points (`grid.csv`).
3. Write the `ini` file.
4. Check where every inlist value comes from (`posydon-grid explain`).
5. Set up the run directory (`posydon-grid setup`).
6. Submit the grid.

<div class='alert alert-info'>

The outputs on this page were made with a stand-in for MESA, so the
compilation is instant, and the paths were shortened to `/home/user/...`.
With your own MESA, `posydon-grid setup` compiles MESA, which takes a few
minutes, and the paths are yours.

</div>

## Before you start

**MESA.** POSYDON v2.0 uses a modified MESA r11701. `posydon-grid setup`
compiles it into the run directory, so it needs to know where MESA and the
MESA SDK are. Put these lines in a file, for example `~/load_MESA`, with
the paths of your installation, and `source` it in every terminal you set
up or run grids from:

```bash
export MESA_DIR=~/MESA/mesa-r11701_witheoschange_andwithreverseMTchange_fiximplicitmdot
export MESASDK_ROOT=~/MESA/mesasdk
source $MESASDK_ROOT/bin/mesasdk_init.sh
export OMP_NUM_THREADS=4
```

<details>
<summary><strong>Which MESA?</strong></summary>

The POSYDON grids use MESA r11701 with fixes to the equation of state, to
reverse mass transfer and to implicit mass transfer. The MESA inlist
parameters are therefore older than the current MESA documentation shows.
Please contact the POSYDON team if you need this version of MESA.

</details>

**The POSYDON inlists.** The inlists, Fortran sources, ZAMS models and
column lists of the POSYDON releases are in the
[POSYDON-MESA-INLISTS](https://github.com/POSYDON-code/POSYDON-MESA-INLISTS)
repository, on its `v2_sources` branch. Clone it once:

```bash
git clone --branch v2_sources https://github.com/POSYDON-code/POSYDON-MESA-INLISTS.git ~/POSYDON-MESA-INLISTS
```

`posydon-grid` reads this clone as it is and never runs git on it, so
updating it is up to you (`git -C ~/POSYDON-MESA-INLISTS pull`). Whatever
state it is in when you set up a grid is written down in the run
directory (step 6).

Make a new, empty folder for this grid and start Jupyter (or a terminal)
there. The cells below run in that folder; a line starting with `!` is a
shell command, which you can also type in a terminal without the `!`.
Check that POSYDON finds the grid types:

In [1]:
!posydon-grid types

HMS-HMS       two hydrogen-rich main-sequence stars
CO-HMS_RLO    a compact object and a hydrogen-rich main-sequence star
CO-HeMS       a compact object and a helium main-sequence star
HeMS-HMS      a helium and a hydrogen-rich main-sequence star
single_HMS    single hydrogen-rich main-sequence stars
single_HeMS   single helium main-sequence stars


## 1. Choose a rerun

A release lives in `r11701/releases/<release>/` of the clone, and its
reruns in `reruns/` below it:

In [2]:
!ls ~/POSYDON-MESA-INLISTS/r11701/releases/v2.0/reruns

development_c4f90ce                  seth_TPAGBwind_runstar
fix_iniHe_pre_rerun5                 seth_dedt_energy_eqn
jeff_new_grid                        seth_dedt_hepulse
main_18710e9                         seth_lbv_thermo_dedt
matthias_LBV_wind                    seth_lbv_thermo_dedt_hepulse
matthias_PISN                        seth_lbv_thermo_dedt_hepulse_77640b5
matthias_fix_initial_Hestar          seth_rerun_egrav_Pgas_dXdt_terms
matthias_lbv_thermo_dedt_noAgeLimit  zepei_fix_implicit
matthias_no_age_limit


Each rerun has a `scenario.ini` that says where it came from, which v2.0
grids used it, and which files it changes. Here we use
`matthias_lbv_thermo_dedt_noAgeLimit`, which most v2.0 HMS-HMS grids were
rerun with:

In [3]:
!cat ~/POSYDON-MESA-INLISTS/r11701/releases/v2.0/reruns/matthias_lbv_thermo_dedt_noAgeLimit/scenario.ini

; Rerun matthias_lbv_thermo_dedt_noAgeLimit of release v2.0 (MESA r11701)

[rerun]
name = matthias_lbv_thermo_dedt_noAgeLimit
release = v2.0
source_branch = matthias_lbv_thermo_dedt_noAgeLimit
source_commit = dcc42e1d6d89fc267826970643e7f4f1bf463171
; grid types the v2 production grids ran it for (ini files)
used_for = CO-HMS_RLO (24), CO-HeMS (8), HMS-HMS (67)

[overlays]
; release file = this rerun's file holding only the keys it
; changes; applied right after the release file, wherever
; the release file is used.
CO-HeMS/binary/inlist1 = CO-HeMS/binary/inlist1
common/binary/inlist1 = common/binary/inlist1
common/binary/inlist2 = common/binary/inlist2

[replace]
; release file = this rerun's file, used instead of it
CO-HeMS/star1_formation/src/run_star_extras.f = CO-HeMS/star1_formation/src/run_star_extras.f
common/binary/src/run_binary_extras.f = common/binary/src/run_binary_extras.f
common/binary/src/run_star_extras.f = common/binary/src/run_star_extras.f


<details>
<summary><strong>Overlays and replacements</strong></summary>

A rerun changes the release in two ways:

- `[overlays]`: an inlist of the rerun holds *only the keys it changes*.
  It is stacked right after the release's inlist, so every other key keeps
  the release's value. Here `common/binary/inlist1` (star 1) and
  `common/binary/inlist2` (star 2) change, among others, `max_age` and
  `thermohaline_coeff`.
- `[replace]`: a whole file (a Fortran source here) is used *instead of*
  the release's.

Have a look at one of the overlays:

```bash
cat ~/POSYDON-MESA-INLISTS/r11701/releases/v2.0/reruns/matthias_lbv_thermo_dedt_noAgeLimit/common/binary/inlist1
```

</details>

## 2. Write the grid points

The grid points are a CSV file with one row per binary. Each column is a
MESA inlist parameter that is set per run. For an HMS-HMS grid these are
the metallicity (`initial_z` and `Zbase`), the two masses in solar masses
(`m1`, `m2`) and the orbital period in days (`initial_period_in_days`).
Here are three binaries at solar metallicity (Z = 0.0142):

In [4]:
import csv

points = [
    # initial_z, Zbase, m1, m2, initial_period_in_days
    (0.0142, 0.0142, 30.0, 21.0, 10.0),
    (0.0142, 0.0142, 30.0, 21.0, 100.0),
    (0.0142, 0.0142, 15.0, 10.5, 10.0),
]
with open("grid.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["initial_z", "Zbase", "m1", "m2",
                     "initial_period_in_days"])
    writer.writerows(points)

In [5]:
!cat grid.csv

<details>
<summary><strong>Metallicity and the ZAMS model</strong></summary>

Both stars start from a zero-age main-sequence (ZAMS) model, which you
choose in the `ini` file. **The metallicity of the grid points and of the
ZAMS model must be the same**, or MESA cannot evolve the stars. The ZAMS
models of the eight POSYDON metallicities are in
`~/POSYDON-MESA-INLISTS/r11701/ZAMS_models/`; their names give the
metallicity: `zams_z1.42m2_y0.2703.data` is Z = 1.42e-2 = 0.0142.

The points of the v2.0 grids were made with
`r11701/running_scripts/parameter_space_v2/create_csv.ipynb` in
POSYDON-MESA-INLISTS.

</details>

## 3. Write the ini file

`posydon-grid init` writes a starter file with every option of a grid
type, each with a short note. Options left at their default start with
`;`:

In [6]:
!posydon-grid init HMS-HMS HMS-HMS.ini

Wrote HMS-HMS.ini; fill in the <...> values.


Open `HMS-HMS.ini` and fill it in. The filled-in file below is the same
file with the `<...>` values set, the rerun switched on, and the options
left at their default removed:

- `[grid]` says what to run: the grid type, the points, where the run
  directory goes (relative to the `ini` file) and the ZAMS model of both
  stars;
- `[scenario]` says which POSYDON inlists to use: the release, the rerun,
  and your clone;
- `[output]` says what MESA writes;
- `[submission]` says how the grid runs: `slurm_array` makes one SLURM
  array task per grid point. Fill in your cluster's partition and account
  (settings you leave out are left out of the SLURM script, so the
  cluster's defaults apply).

In [7]:
%%writefile HMS-HMS.ini
[grid]
type = HMS-HMS
mode = fixed
points = grid.csv
run_directory = run
zams = ~/POSYDON-MESA-INLISTS/r11701/ZAMS_models/zams_z1.42m2_y0.2703.data

[scenario]
release = v2.0
rerun = matthias_lbv_thermo_dedt_noAgeLimit
inlist_repo = ~/POSYDON-MESA-INLISTS

[output]
history_interval = 1
star1 = history, final_model
star2 = history, final_model
binary = history

[submission]
backend = slurm_array
partition = my_partition
account = my_account
walltime = 2-00:00:00
cpus_per_task = 4

Overwriting HMS-HMS.ini


## 4. Check where every value comes from

Before writing anything, ask `posydon-grid` how it will build the inlists.
`posydon-grid explain` lists, for every namelist, each key that POSYDON
(or you) set, with the file it comes from and the values it overrides. It
writes nothing. The list is long, so save it to a file:

In [8]:
!posydon-grid explain HMS-HMS.ini > explain.txt
!grep '^[a-z]' explain.txt

binary_controls
binary_job
star1_controls
star1_job
star2_controls
star2_job


The lines without indentation are the namelists. Now look up a key the
rerun changes, `max_age`:

In [9]:
!grep -A2 '^  max_age ' explain.txt

  max_age = -1  [posydon_inlists] r11701/releases/v2.0/reruns/matthias_lbv_thermo_dedt_noAgeLimit/common/binary/inlist1
      over 1.4e10  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist1
      over 1d36  [mesa_inlists] r11701/mesa/default/defaults/star/controls.defaults
--
  max_age = -1  [posydon_inlists] r11701/releases/v2.0/reruns/matthias_lbv_thermo_dedt_noAgeLimit/common/binary/inlist2
      over 1.4e10  [posydon_inlists] r11701/releases/v2.0/common/binary/inlist2
      over 1d36  [mesa_inlists] r11701/mesa/default/defaults/star/controls.defaults


Each block reads from top to bottom as "this value, from this file,
over that value, from that file". For both stars (`inlist1` is star 1,
`inlist2` star 2) the rerun's `max_age = -1` (no age limit) wins over the
release's `1.4e10`, which won over MESA's default. The labels are the
layers:

| Layer | What it is |
|---|---|
| `[mesa_inlists]` | MESA's defaults, of the MESA installation of the release |
| `[posydon_inlists]` | the release's inlists, and the rerun's on top |
| `[user_inlists]` | your own inlists, if you add any (see [Stack your own inlist on a POSYDON scenario](stack_own_inlist.ipynb)) |

The keys the setup writes itself (the ZAMS model, the output options and
the wiring between the inlists) go on top of all of them.

## 5. Set up the run directory

`posydon-grid setup` writes the run directory: it compiles MESA, writes
the inlists, copies the column lists and Fortran sources, and writes the
SLURM scripts:

In [10]:
!posydon-grid setup HMS-HMS.ini

In [11]:
!ls run

binary                      run_grid.sh
cleanup.slurm               setup_record.ini
column_lists                star1
job_array_grid_submit.slurm star2
mk


<details>
<summary><strong>What each file is</strong></summary>

- `binary/`, `star1/`, `star2/`: the MESA work folders, with the compiled
  executables, the written inlists (`binary/inlist_project`,
  `binary/inlist1`, `binary/inlist2`) and the Fortran sources (`src/`).
- `column_lists/`: the history and profile columns MESA writes.
- `mk`: the script that compiled MESA.
- `job_array_grid_submit.slurm`: the SLURM job array, one task per grid
  point.
- `cleanup.slurm`: a job that runs after the grid, to compress the output.
- `run_grid.sh`: submits both.
- `setup_record.ini`: what the run directory was set up from (below).

</details>

The written inlists are complete: every MESA key is written, with its
final value. For example, star 1's `max_age`:

In [12]:
!grep -n 'max_age ' run/binary/inlist1

84:	max_age = -1
86:	num_adjusted_dt_steps_before_max_age = 0
87:	dt_years_for_steps_before_max_age = 1d6


### The record

`run/setup_record.ini` writes down what the run directory was set up
from: the command, the POSYDON and MESA used, the inlist repository as it
was read, and for every inlist the files stacked into it. Its
`[scenario]` section names the release and rerun, and the commit of your
clone:

In [13]:
!sed -n '/^\[scenario\]/,/^$/p' run/setup_record.ini

[scenario]
release = v2.0
rerun = matthias_lbv_thermo_dedt_noAgeLimit
mesa_version = r11701
mesa = default
inlist_repo = /home/user/POSYDON-MESA-INLISTS
inlist_remote = https://github.com/POSYDON-code/POSYDON-MESA-INLISTS.git
inlist_commit = 115de83c33bad5013cb09837fcc679d1d45e816b
inlist_dirty = False
inlist_fingerprint = 0df9a0d5a2161ceea9d4c4fd465abc6236b83fc3793147647c5bc17cefe6a0c4
inlist_files = 17



`inlist_dirty = False` says the files used are the ones of that commit,
and `inlist_fingerprint` is a checksum of those files: two grids with the
same fingerprint were built from identical files, whatever folder or
branch they came from. The `[inlists]` section lists, for each namelist,
the files stacked into it, lowest layer first:

In [14]:
!sed -n '/^\[inlists\]/,/^$/p' run/setup_record.ini | head -12

[inlists]
binary/inlist_project &binary_controls = 
	mesa_inlists     r11701/mesa/default/defaults/binary/binary_controls.defaults
	posydon_inlists  r11701/releases/v2.0/common/binary/inlist_project
binary/inlist_project &binary_job = 
	mesa_inlists     r11701/mesa/default/defaults/binary/binary_job.defaults
	posydon_inlists  r11701/releases/v2.0/common/binary/inlist_project
binary/inlist1 &controls = 
	mesa_inlists     r11701/mesa/default/defaults/star/controls.defaults
	posydon_inlists  r11701/releases/v2.0/common/binary/inlist1
	posydon_inlists  r11701/releases/v2.0/reruns/matthias_lbv_thermo_dedt_noAgeLimit/common/binary/inlist1
binary/inlist1 &star_job = 


## 6. Submit the grid

On the cluster, go into the run directory and submit:

```bash
cd run
./run_grid.sh
```

This submits the job array and, after it, the cleanup job:

```
job_array_grid_submit.slurm submitted as 1234567
cleanup.slurm submitted as 1234568
```

Each grid point runs in its own folder in `run/`, named after its
parameters, for example
`Zbase_0.0142_m1_30.0000_m2_21.0000_initial_z_1.4200e-02_initial_period_in_days_1.0000e+01_grid_index_0`.
MESA's screen output is in `out.txt` there (compressed to `out.txt.gz` by
the cleanup job). Follow the jobs with `squeue -u $USER`.

When the grid has finished, turn it into a `PSyGrid` with the
[post-processing pipeline](../../generating-datasets/generating-datasets.rst).

## See also

- [Set up a single-HMS grid with a rerun, pinned to one version](single_hms_rerun.ipynb)
- [Stack your own inlist on a POSYDON scenario](stack_own_inlist.ipynb)
- [Run a few points on your laptop](shell_laptop.ipynb)
- [The posydon-grid command](../../../components-overview/mesa_grids/posydon_grid.rst)
  and [the ini file](../../../components-overview/mesa_grids/inifile.rst)